# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsimaZaheer/Task1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1

The paper reports that cannibalizing queries represent 703.2K queries, with 73.2M impressions and 274.4K clicks considered at risk. The paper groups these cases by severity and uses the result to identify portfolio-level structural risks.

My methodology question:
How is a cannibalization case defined and labeled? I would want to understand whether the label comes directly from observed search data or from a defined rule based on overlapping queries/pages. This matters because the definition affects how strongly the finding can be interpreted.


### Finding 2

The paper reports that 58.2% of the analyzed portfolio is confirmed indexed, while 41.8% is classified as never indexed. It uses this result to highlight indexing coverage as an important portfolio-level issue.

My methodology question:
How was the indexing status validated, and does the available data cover the full portfolio consistently? I would want to know whether the validation process and data coverage support extending this finding to the whole portfolio.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

### Interpretation

Under the client-grouped validation split, the Random Forest achieved a measured ROC AUC of 0.897 and Average Precision of 0.908. These results are slightly lower than the Week-5 reported results of 0.910 ROC AUC and 0.919 Average Precision.

The decrease suggests that validation design affects the measured result. The grouped split is more conservative because pages from the same client are kept entirely in either training or testing.

The model still shows useful measured discrimination on this split, but the result should be treated as directional and decision-support evidence rather than proof of future performance.

In [1]:
!git clone https://github.com/AsimaZaheer/Task1.git

Cloning into 'Task1'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 158 (delta 65), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.87 MiB | 14.52 MiB/s, done.
Resolving deltas: 100% (65/65), done.


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/Task1/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
print("Unique clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Unique clients: 32


In [4]:
# Target: observed declining trend
y = (df["trend_direction"] == "down").astype(int)

# Features available before the decision moment
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[feature_cols].copy()

print("Feature matrix shape:", X.shape)
print("\nTarget distribution:")
print(y.value_counts())

print("\nDeclining rate:", round(y.mean(), 3))

Feature matrix shape: (30000, 26)

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Declining rate: 0.542


In [5]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("Overlapping clients:",
      len(set(groups_train) & set(groups_test)))

print("Training declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Overlapping clients: 0
Training declining rate: 0.55
Test declining rate: 0.511


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]

rf_roc_auc = roc_auc_score(y_test, rf_prob)
rf_ap = average_precision_score(y_test, rf_prob)

print("Honest grouped validation")
print("-------------------------")
print("ROC AUC:", round(rf_roc_auc, 3))
print("Average Precision:", round(rf_ap, 3))

Honest grouped validation
-------------------------
ROC AUC: 0.897
Average Precision: 0.908


In [7]:
comparison = pd.DataFrame({
    "Validation": [
        "Week-5 reported result",
        "ML-09 honest grouped validation"
    ],
    "ROC AUC": [
        0.910,
        rf_roc_auc
    ],
    "Average Precision": [
        0.919,
        rf_ap
    ]
})

display(comparison.round(3))

,Validation,ROC AUC,Average Precision
0,Week-5 reported result,0.910,0.919
1,ML-09 honest grouped validation,0.897,0.908


## 3. Leakage audit

### Leakage audit conclusion

The final model uses 26 features. The target-related fields `trend_direction` and `trend_pct` were explicitly excluded from the feature set.

The audit therefore confirms that these two known target-derived fields were not used as model inputs. Identifier fields were also not included as predictive features.

The remaining features represent observed search, engagement, content-age, and performance signals available in the dataset. This audit does not prove that every feature is free from all possible temporal leakage, so the result should be interpreted as a documented feature-level leakage check rather than a guarantee of leakage-free data generation.

In [8]:
# Leakage audit: check whether target-derived fields are in the final feature set

target_related = [
    "trend_direction",
    "trend_pct"
]

print("Target-related fields:")
for col in target_related:
    print(
        f"{col}:",
        "IN FEATURE SET" if col in feature_cols else "NOT IN FEATURE SET"
    )

print("\nFinal feature count:", len(feature_cols))
print("\nFinal features:")
print(feature_cols)

Target-related fields:
trend_direction: NOT IN FEATURE SET
trend_pct: NOT IN FEATURE SET

Final feature count: 26

Final features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate']


In [9]:
# Classify the final features for leakage audit

leakage_audit = []

for col in feature_cols:
    if col in ["trend_direction", "trend_pct"]:
        verdict = "LEAKAGE - exclude"
        reason = "Derived from the target trend"
    elif col in ["content_id", "client_id"]:
        verdict = "IDENTIFIER - exclude"
        reason = "Identifier, not a predictive feature"
    else:
        verdict = "RETAIN"
        reason = "Observed feature used for the model"

    leakage_audit.append({
        "feature": col,
        "verdict": verdict,
        "reason": reason
    })

leakage_audit = pd.DataFrame(leakage_audit)

display(leakage_audit)

,feature,verdict,reason
0,search_volume,RETAIN,Observed feature used for the model
1,competition,RETAIN,Observed feature used for the model
2,cpc,RETAIN,Observed feature used for the model
3,word_count,RETAIN,Observed feature used for the model
4,char_count,RETAIN,Observed feature used for the model
5,impressions_90d,RETAIN,Observed feature used for the model
6,clicks_90d,RETAIN,Observed feature used for the model
7,pageviews_90d,RETAIN,Observed feature used for the model
8,sessions_90d,RETAIN,Observed feature used for the model
9,users_90d,RETAIN,Observed feature used for the model


## 4. Claim rewrite


My original claim was that the Random Forest performed better than the baseline and could help identify pages that need refreshing.

A safer claim is:

"On the evaluated client-grouped test split, the Random Forest showed stronger measured performance than the Week-4 baseline for the observed declining proxy. The results are directional and support using the model as decision-support for prioritizing human review. They do not prove that the model predicts future Google performance or that refreshing a flagged page will cause recovery."

In [10]:
print("Week-5 ROC AUC:", 0.910)
print("ML-09 grouped ROC AUC:", round(rf_roc_auc, 3))

print("\nWeek-5 Average Precision:", 0.919)
print("ML-09 grouped Average Precision:", round(rf_ap, 3))

print("\nClaim status: Directional / Decision-support")

Week-5 ROC AUC: 0.91
ML-09 grouped ROC AUC: 0.897

Week-5 Average Precision: 0.919
ML-09 grouped Average Precision: 0.908

Claim status: Directional / Decision-support


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.